# 12 — Multi-head Latent Attention (MLA)

**Paper:** DeepSeek-V2 §2.1 — [arxiv](https://arxiv.org/html/2405.04434)

## GPT-2 MHA (notebook 4)

Each token caches **full** `K` and `V` per head → large KV cache at long context.

## MLA idea (one sentence)

Compress `(K,V)` into a **small latent vector** `c_kv` per token; reconstruct K/V when needed.

At inference you mainly store `c_kv` (size `kv_lora_rank`), not `2 × n_head × head_dim`.


In [ ]:
import torch
from llmc.deepseek_v2 import DeepSeekV2Config, MultiHeadLatentAttention, RMSNorm

cfg = DeepSeekV2Config.tiny(vocab_size=128, block_size=32)
x = torch.randn(2, 16, cfg.n_embd)
attn = MultiHeadLatentAttention(cfg)

# Pre-norm like the real block (RMSNorm is used inside the block; here we show raw attn)
y = attn(x)

print("input:", tuple(x.shape))
print("output:", tuple(y.shape))
print("latent c_kv cached:", tuple(attn.last_c_kv.shape), "  # (B, T, kv_lora_rank)")


In [ ]:
# Compare KV cache footprint (educational estimate, float32)
from llmc.deepseek_v2 import DeepSeekV2

model = DeepSeekV2(cfg)
print("MLA cache bytes/token (all layers):", model.kv_cache_bytes_per_token())
print("MHA cache bytes/token (hypothetical GPT-2):", model.mha_kv_cache_bytes_per_token())
print("ratio MHA/MLA:", model.mha_kv_cache_bytes_per_token() / model.kv_cache_bytes_per_token())


## Step-by-step (read the code)

1. `w_dkv(x)` → **`c_kv`** with shape `(B, T, kv_lora_rank)` — this is the **compressed cache**.
2. `w_uk(c_kv)` and `w_uv(c_kv)` → full keys and values per head (reconstructed when needed).
3. Causal softmax attention — **same math as notebook 4**, different way to build K/V.

## C port (piece 1 — done)

```bash
cd c && make test_mla && ./bin/test_mla
```

Open **`c/deepseek_v2/mla.c`** — comments map line-by-line to `MultiHeadLatentAttention` in `llmc/deepseek_v2.py`.

In **`vendor/llm.c/train_gpt2.c`**, search for `attention_forward`; MLA replaces that block once we wire weights into a full trainer.
